# Period-independent wavelength advisory workflow

This tutorial replaces the older automatic-model-selection notebook.
It shows how to inspect wavelength-dependent structure, build an
advisory parameter plan, and prepare LPV-relevant model/kernel
configurations **without automatically selecting a model**.

The historical filename `tutorial_model_selection.ipynb` was retired
because the current workflow is advisory: `advisory_only` remains true
and `selected_model` remains `None`.


## What this tutorial does

The preparation path below:

1. creates a deterministic four-band light curve with
   wavelength-dependent median flux and amplitude;
2. computes period-independent robust per-band summaries;
3. builds advisory parameter suggestions;
4. prepares model/kernel configs for `2DWavelengthDependent`,
   `2DDustMean`, `2DPowerLawMean`, and the optional `2D` baseline; and
5. leaves GP fitting disabled unless `RUN_FITS` is changed explicitly.

It does not infer a period, use temporal consensus, fit in log-flux
space, or install a winning model.


In [ ]:
import numpy as np
import torch

from pgmuvi.dtypes import DEFAULT_DTYPE
from pgmuvi.lightcurve import Lightcurve

SEED = 20260715
RUN_FITS = False
rng = np.random.default_rng(SEED)

print(f"PGMUVI default dtype: {DEFAULT_DTYPE}")
print(f"advisory seed: {SEED}")
print(f"RUN_FITS: {RUN_FITS}")


## 1. Build a deterministic multiwavelength light curve

Numeric wavelengths are stored in the second input coordinate, while
human-readable band names are stored separately. The injected temporal
signal is only a convenient way to create realistic per-band
distributions; the first diagnostic deliberately ignores its period.


In [ ]:
wavelengths = np.array([0.80, 1.25, 2.20, 3.40])
band_names = np.array(["I", "J", "K", "W1"])
counts = np.array([48, 52, 56, 60])
injected_period = 420.0

x_blocks = []
y_blocks = []
yerr_blocks = []
band_labels = []

median_fluxes = np.array([8.0, 12.0, 20.0, 27.0])
half_amplitudes = np.array([0.8, 1.1, 1.5, 1.8])

for wavelength, band, n_points, median_flux, half_amplitude in zip(
    wavelengths,
    band_names,
    counts,
    median_fluxes,
    half_amplitudes,
    strict=True,
):
    time = np.sort(rng.uniform(0.0, 1600.0, int(n_points)))
    phase = 2.0 * np.pi * time / injected_period
    uncertainty = np.full(int(n_points), 0.12 + 0.02 * wavelength)
    flux = (
        median_flux
        + half_amplitude * np.sin(phase)
        + 0.12 * half_amplitude * np.cos(2.0 * phase)
        + rng.normal(0.0, uncertainty)
    )

    x_blocks.append(
        np.column_stack([time, np.full(int(n_points), wavelength)])
    )
    y_blocks.append(flux)
    yerr_blocks.append(uncertainty)
    band_labels.extend([band] * int(n_points))

x = np.vstack(x_blocks)
y = np.concatenate(y_blocks)
yerr = np.concatenate(yerr_blocks)

lc = Lightcurve(
    torch.as_tensor(x, dtype=DEFAULT_DTYPE),
    torch.as_tensor(y, dtype=DEFAULT_DTYPE),
    yerr=torch.as_tensor(yerr, dtype=DEFAULT_DTYPE),
    band=np.asarray(band_labels, dtype=str),
    max_samples=None,
)

print(f"rows: {len(lc.ydata)}")
print(f"input dimensions: {lc.ndim}")
print(f"wavelengths: {wavelengths.tolist()}")
print(f"band counts: {counts.tolist()}")
print(f"all fluxes positive: {bool(np.all(y > 0.0))}")


## 2. Period-independent wavelength structure

`diagnose_period_independent_wavelength_structure()` uses only robust
per-band flux distributions. It does not use Lomb--Scargle peaks, ACF
peaks, a consensus frequency, phase folding, or a GP fit. The
central-95% amplitude proxy is based on `q02_5` and `q97_5`; it is
descriptive and is not itself a recovered pulsation amplitude.


In [ ]:
diagnostics = lc.diagnose_period_independent_wavelength_structure(
    min_points_per_band=20
)
diagnostic_summary = diagnostics["summary"]

print(f"kind: {diagnostics['kind']}")
print(f"period independent: {diagnostics['is_period_independent']}")
print(
    "uses temporal consensus: "
    f"{diagnostic_summary['uses_temporal_consensus']}"
)
print(
    "uses period or frequency: "
    f"{diagnostic_summary['uses_period_or_frequency']}"
)
print(f"usable bands: {diagnostic_summary['n_usable_bands']}")
print(
    "median trend: "
    f"{diagnostic_summary['median_flux_monotonicity_class']}"
)
print(
    "central-95% amplitude trend: "
    f"{diagnostic_summary['raw_half_amplitude_q02_5_q97_5_monotonicity_class']}"
)


In [ ]:
band_rows = []
for row in diagnostics["band_table"]:
    flux_summary = row["period_independent_flux_summary"]
    band_rows.append(
        {
            "wavelength": row["wavelength"],
            "labels": row["band_labels"],
            "n_points": row["n_points"],
            "median_flux": flux_summary["median_flux"],
            "half_amplitude_q02_5_q97_5": flux_summary[
                "raw_half_amplitude_q02_5_q97_5"
            ],
        }
    )

for row in band_rows:
    print(row)


The monotonicity classes and robust amplitudes are evidence for
follow-up, not hard model exclusions. Uneven phase coverage, noise,
multiple periods, and non-stationary variability can all affect these
summaries.


## 3. Build an advisory parameter plan

The parameter plan translates the distribution summaries into ranked
model-family suggestions and model-specific initialization/constraint
metadata. It does not mutate the light curve, register constraints, or
apply initial values.


In [ ]:
parameter_plan = lc.build_period_independent_wavelength_parameter_plan(
    diagnostics_report=diagnostics
)
ranked_models = [
    entry["model"] for entry in parameter_plan["ranked_candidates"]
]

print(f"kind: {parameter_plan['kind']}")
print(f"advisory only: {parameter_plan['advisory_only']}")
print(
    "primary recommendation: "
    f"{parameter_plan['primary_recommended_model']}"
)
print(f"ranked models: {ranked_models}")
print(
    "automatic initialization applied: "
    f"{parameter_plan['automatic_initialization_applied']}"
)
print(
    "automatic constraints applied: "
    f"{parameter_plan['automatic_constraints_applied']}"
)


## 4. Prepare model/kernel configs

LPV-relevant separable families use
`time_kernel_type="quasi_periodic"` with consensus period handoff. The
optional full `2D` baseline retains its spectral-mixture default.
Parameter suggestions remain metadata and are not inserted into
`fit_kwargs`.


In [ ]:
config_report = lc.build_period_independent_wavelength_model_kernel_configs(
    parameter_plan=parameter_plan,
    include_2d_baseline=True,
    base_fit_kwargs={
        "training_iter": 100,
        "miniter": 20,
        "fit_strategy": "consensus",
        "learn_additional_noise": True,
        "verbose": False,
    },
)
config_models = [
    entry["model"]
    for entry in config_report["model_kernel_configs"]
]

print(f"kind: {config_report['kind']}")
print(f"configs run fits: {config_report['runs_fits']}")
print(f"model/kernel configs: {config_models}")
for entry in config_report["model_kernel_configs"]:
    print(
        entry["rank"],
        entry["model"],
        entry["fit_kwargs"].get(
            "time_kernel_type", "spectral_mixture default"
        ),
        entry["parameter_suggestions_applied"],
    )


`2DSeparable` is a useful direct product-kernel comparison, but it is
not automatically inserted into this LPV advisory list. Add and fit it
explicitly when the scientific question calls for that covariance
structure.


## 5. Optional model/kernel-config fits

The full advisory workflow fits isolated copies of the light curve,
scores training-residual diagnostics, and can format a comparison
report. It can be slow and can fail for consensus, numerical, or
input-validation reasons, so this tutorial leaves it disabled by
default.


In [ ]:
workflow = None
if RUN_FITS:
    workflow = lc.run_period_independent_wavelength_advisory_workflow(
        model_kernel_config_report=config_report,
        stop_on_error=False,
        make_text_report=True,
        make_plots=False,
    )
    print(workflow["text_report"])
else:
    print("PREPARE ONLY: advisory model/kernel fits were not started.")


## 6. Interpret the advisory result

When fits are enabled, inspect `quality_report`, `fallback_report`,
per-config failures, residual diagnostics, ARD scale-ceiling fields,
and the actual fitted light curves. `top_ranked_model` is a triage
result. It is not equivalent to scientific validation, held-out
predictive superiority, or a selected production model.


In [ ]:
if workflow is None:
    print(
        "selected_model remains unavailable because no fits were run."
    )
else:
    print(f"top ranked model: {workflow['top_ranked_model']}")
    print(f"selected model: {workflow['selected_model']}")
    print(f"advisory only: {workflow['advisory_only']}")
    print(
        "fallback diagnostics available: "
        f"{workflow['fallback_report']['available']}"
    )


## 7. What remains future work

The current advisory scores are based on training-residual diagnostics,
not held-out prediction, posterior predictive checks, marginal
likelihood, or automatic evidence-based selection. Multi-periodic
support, non-monotonic wavelength kernels, physically motivated
wavelength-dependence kernels, and a validated automatic-selection
framework remain future work.

For the full API and exported artifacts, continue with the
[single-source wavelength advisory guide](../howto/wavelength_advisory.rst),
the [model-family guide](../howto/wavelength_models.rst), and the
[batch walkthrough](../howto/wavelength_advisory_batch.rst).
